# MNIST Lightning Module

#| default_exp modules.mnist_litmodule

This notebook defines the training module for MNIST using PyTorch Lightning.

In [ ]:
#| default_exp modules.mnist_litmodule

In [ ]:
#| export
from typing import Any
from fastcore.utils import patch
from src.lightning_config import lightning_config

import torch
from pytorch_lightning import LightningModule
from torchmetrics import MaxMetric
from torchmetrics.classification.accuracy import Accuracy

from src.models.simpledense import SimpleDenseNet


@lightning_config(init_args={
    "net": {
        "class_path": "src.models.simpledense.SimpleDenseNet",
        "init_args": {
            "input_size": 784,
            "lin1_size": 256,
            "lin2_size": 256,
            "lin3_size": 256,
            "output_size": 10
        }
    }
})
class MNISTLitModule(LightningModule):
    """Example of LightningModule for MNIST classification."""

    def __init__(
        self,
        net: torch.nn.Module,
        lr: float = 0.001,
        weight_decay: float = 0.0005,
    ):
        super().__init__()
        self.save_hyperparameters(logger=False, ignore=["net"])
        self.net = net
        self.criterion = torch.nn.CrossEntropyLoss()
        self.train_acc = Accuracy(task='multiclass', num_classes=10)
        self.val_acc = Accuracy(task='multiclass', num_classes=10)
        self.test_acc = Accuracy(task='multiclass', num_classes=10)
        self.val_acc_best = MaxMetric()

    def forward(self, x: torch.Tensor):
        return self.net(x)

    def on_train_start(self):
        self.val_acc_best.reset()

    def step(self, batch: Any):
        x, y = batch
        logits = self.forward(x)
        loss = self.criterion(logits, y)
        preds = torch.argmax(logits, dim=1)
        return loss, preds, y

## Training Step

The training step computes the loss and accuracy on a batch of training data.

In [ ]:
#| export
@patch
def training_step(self: MNISTLitModule, batch: Any, batch_idx: int):
    loss, preds, targets = self.step(batch)

    # log train metrics
    acc = self.train_acc(preds, targets)
    self.log("train/loss", loss, on_step=False, on_epoch=True, prog_bar=False)
    self.log("train/acc", acc, on_step=False, on_epoch=True, prog_bar=True)

    return {"loss": loss, "preds": preds, "targets": targets}

@patch
def on_training_epoch_end(self: MNISTLitModule):
    self.train_acc.reset()

## Validation Step

The validation step computes loss and accuracy on validation data, and tracks the best validation accuracy seen so far.

In [ ]:
#| export
@patch
def validation_step(self: MNISTLitModule, batch: Any, batch_idx: int):
    loss, preds, targets = self.step(batch)

    # log val metrics
    acc = self.val_acc(preds, targets)
    self.log("val/loss", loss, on_step=False, on_epoch=True, prog_bar=False)
    self.log("val/acc", acc, on_step=False, on_epoch=True, prog_bar=True)

    return {"loss": loss, "preds": preds, "targets": targets}

@patch
def on_validation_epoch_end(self: MNISTLitModule):
    acc = self.val_acc.compute()  # get val accuracy from current epoch
    self.val_acc_best.update(acc)
    self.log(
        "val/acc_best", self.val_acc_best.compute(), on_epoch=True, prog_bar=True
    )
    self.val_acc.reset()  # reset val accuracy for next epoch

## Test Step

The test step evaluates the model on test data.

In [ ]:
#| export
@patch
def test_step(self: MNISTLitModule, batch: Any, batch_idx: int):
    loss, preds, targets = self.step(batch)

    # log test metrics
    acc = self.test_acc(preds, targets)
    self.log("test/loss", loss, on_step=False, on_epoch=True)
    self.log("test/acc", acc, on_step=False, on_epoch=True)

    return {"loss": loss, "preds": preds, "targets": targets}

@patch
def on_test_epoch_end(self: MNISTLitModule):
    self.test_acc.reset()

## Optimizer Configuration

Configure the Adam optimizer with the specified learning rate and weight decay.

In [ ]:
#| export
@patch
def configure_optimizers(self: MNISTLitModule):
    """Choose what optimizers and learning-rate schedulers to use in your optimization.
    Normally you'd need one. But in the case of GANs or similar you might have multiple.

    See examples here:
        https://pytorch-lightning.readthedocs.io/en/latest/common/lightning_module.html#configure-optimizers
    """
    return torch.optim.Adam(
        params=self.parameters(),
        lr=self.hparams.lr,
        weight_decay=self.hparams.weight_decay,
    )